<a href="https://colab.research.google.com/github/Annsujitra001/Lab-4/blob/main/Lab_4_Geographic_Modeling.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import ee
import geemap

ee.Authenticate()
ee.Initialize(project='ee-annsujitra001')

In [ ]:
# -----------------------
# ROI: Nakhon Si Thammarat
# -----------------------
provinces = ee.FeatureCollection("FAO/GAUL/2015/level1")

roi = provinces.filter(ee.Filter.eq('ADM1_NAME', 'Nakhon Si Thammarat'))

In [ ]:
# -----------------------
# Load Data
# -----------------------

dem = ee.Image("USGS/SRTMGL1_003")

rain = ee.ImageCollection("ECMWF/ERA5/MONTHLY") \
    .select("total_precipitation") \
    .filterDate("2020-01-01", "2022-12-31") \
    .mean()

ndvi = ee.ImageCollection("MODIS/006/MOD13A2") \
    .select("NDVI") \
    .filterDate("2020-01-01", "2022-12-31") \
    .mean() \
    .multiply(0.0001) \
    .rename("NDVI")

provinces = ee.FeatureCollection("FAO/GAUL/2015/level1")

roi = provinces.filter(ee.Filter.eq('ADM1_NAME', 'Nakhon Si Thammarat'))

# -----------------------
# ใช้ ROI แทน dem.geometry()
# -----------------------

rain_value = rain.reduceRegion(
    reducer=ee.Reducer.mean(),
    geometry=roi,   # ✅ แก้ตรงนี้
    scale=1000,
    maxPixels=1e9
)

print(rain_value.getInfo())

river = ee.Image("WWF/HydroSHEDS/15ACC")

# -----------------------
# Derive Factors (ต้องมาก่อน!)
# -----------------------

# DEM
dem = ee.Image("USGS/SRTMGL1_003")

# Slope (ต้องสร้างก่อนใช้)
slope = ee.Terrain.slope(dem)

# Distance to river
river_mask = river.gt(100)
distance = river_mask.fastDistanceTransform().sqrt()

# -----------------------
# Normalize
# -----------------------

# clip ทุกตัว
dem = dem.clip(roi)
rain = rain.clip(roi)
slope = slope.clip(roi)
distance = distance.clip(roi)

def normalize(img):
    stats = img.reduceRegion(
        reducer=ee.Reducer.minMax(),
        geometry=roi,
        scale=1000,
        maxPixels=1e13
    )

    minv = ee.Number(stats.values().get(0))
    maxv = ee.Number(stats.values().get(1))

    return img.subtract(minv).divide(maxv.subtract(minv))

# ค่อย normalize
slope_n = normalize(slope)
elev_n = normalize(dem)
rain_n = normalize(rain)
dist_n = normalize(distance)

/usr/local/lib/python3.12/dist-packages/ee/deprecation.py:207: DeprecationWarning: 

Attention required for MODIS/006/MOD13A2! You are using a deprecated asset.
To make sure your code keeps working, please update it.
Learn more: https://developers.google.com/earth-engine/datasets/catalog/MODIS_006_MOD13A2

  warnings.warn(warning, category=DeprecationWarning)


{'total_precipitation': 0.10517235188941924}


In [ ]:
# -----------------------
# Weighted Model
# -----------------------

flood_risk = (rain_n.multiply(0.45)
              .add(slope_n.multiply(0.25))
              .add(elev_n.multiply(0.2))
              .add(dist_n.multiply(0.1)))

# -----------------------
# Classification
# -----------------------

flood_class = flood_risk.expression(
    "(b(0) < 0.4) ? 1"
    ": (b(0) < 0.7) ? 2"
    ": 3"
).rename("risk").toByte()

# -----------------------
# จังหวัดนครศรีธรรมราช
# -----------------------

thai = ee.FeatureCollection("FAO/GAUL/2015/level1")

nst = thai.filter(ee.Filter.eq("ADM1_NAME", "Nakhon Si Thammarat"))

# -----------------------
# Clip ให้เหลือเฉพาะจังหวัด
# -----------------------

flood_clip = flood_class.clip(nst.geometry())

# -----------------------
# Map
# -----------------------

Map = geemap.Map()
Map.centerObject(nst, 8)

# ✅ ใช้ flood_clip (สำคัญมาก)
Map.addLayer(
    flood_clip,
    {
        "min": 1,
        "max": 3,
        "palette": ["green", "yellow", "red"]
    },
    "Flood Risk (NST)"
)

# -----------------------
# เพิ่มขอบเขตจังหวัด
# -----------------------

Map.addLayer(
    nst.style(**{
        "color": "black",
        "fillColor": "00000000",
        "width": 2
    }),
    {},
    "Province Boundary"
)

# -----------------------
# Legend (อ่านง่าย)
# -----------------------

legend = [
    ("Low Risk", "green"),
    ("Moderate Risk", "yellow"),
    ("High Risk", "red")
]

Map.add_legend(title="Flood Risk Level", legend=legend)

Map

Map(center=[8.375329278730176, 99.78833952396523], controls=(WidgetControl(options=['position', 'transparent_b…

In [ ]:
task = ee.batch.Export.image.toDrive(
    image=flood_clip.visualize(
        min=1,
        max=3,
        palette=['#2ecc71', '#f1c40f', '#e74c3c']  # เขียว เหลือง แดง (สวยขึ้น)
    ),
    description='Flood_Risk_Map_NST',
    folder='GEE',
    fileNamePrefix='Flood_Risk_Map_NakhonSiThammarat',
    region=nst.geometry(),   # ✅ ขอบเขตจังหวัด
    scale=100,
    maxPixels=1e13
)

task.start()

In [ ]:
import ee
import geemap

ee.Authenticate()
ee.Initialize(project='ee-annsujitra001')

# -----------------------
# ROI: Nakhon Si Thammarat
# -----------------------
provinces = ee.FeatureCollection("FAO/GAUL/2015/level1")

roi = provinces.filter(ee.Filter.eq('ADM1_NAME', 'Nakhon Si Thammarat'))

# -----------------------n# Normalized inputs (0-1)
# -----------------------
slope = ee.Image("USGS/SRTMGL1_003").select('elevation')
rain = ee.ImageCollection("ECMWF/ERA5/MONTHLY") \
    .select("total_precipitation") \
    .filterDate("2020-01-01", "2022-12-31") \
    .mean()
ndvi = ee.ImageCollection("MODIS/006/MOD13A2") \
    .select("NDVI") \
    .filterDate("2020-01-01", "2022-12-31") \
    .mean() \
    .multiply(0.0001)

# normalize function
def normalize(img):
    img_clipped = img.clip(roi) # Clip image to ROI first
    band_name = ee.String(img.bandNames().get(0)) # Get the name of the first band

    stats = img_clipped.reduceRegion(
        reducer=ee.Reducer.minMax(),
        geometry=roi,
        scale=1000,
        maxPixels=1e9
    )

    minv = ee.Number(stats.get(band_name.cat('_min')))
    maxv = ee.Number(stats.get(band_name.cat('_max')))

    diff = maxv.subtract(minv)

    # Conditionally normalize. If min==max, return a constant image (e.g., 0.5) to avoid division by zero.
    normalized_img = ee.Algorithms.If(
        diff.eq(0),
        ee.Image(0.5).float().rename([band_name]), # Return constant 0.5 if no variability
        img_clipped.subtract(minv).divide(diff).rename([band_name])
    )
    return ee.Image(normalized_img)

slope_n = normalize(slope)
rain_n = normalize(rain)
ndvi_n = normalize(ndvi)

# -----------------------n# Base weights
# -----------------------
w_slope = 0.4
w_rain = 0.4
w_ndvi = 0.2

# -----------------------n# Base model
# -----------------------
risk_base = slope_n.multiply(w_slope)\
    .add(rain_n.multiply(w_rain))\
    .add(ndvi_n.multiply(w_ndvi))

# -----------------------n# Sensitivity (+20% rain weight)
# -----------------------
w_rain_high = w_rain * 1.2
w_rain_high = w_rain_high / (w_slope + w_rain_high + w_ndvi)

risk_rain_high = slope_n.multiply(w_slope)\
    .add(rain_n.multiply(w_rain_high))\
    .add(ndvi_n.multiply(w_ndvi))

# -----------------------n# Sensitivity (-20% rain weight)
# -----------------------
w_rain_low = w_rain * 0.8
w_rain_low = w_rain_low / (w_slope + w_rain_low + w_ndvi)

risk_rain_low = slope_n.multiply(w_slope)\
    .add(rain_n.multiply(w_rain_low))\
    .add(ndvi_n.multiply(w_ndvi))

# -----------------------n# Difference map
# -----------------------
diff_high = risk_rain_high.subtract(risk_base)
diff_low = risk_rain_low.subtract(risk_base)

Map = geemap.Map()
Map.addLayer(risk_base, {}, "Base Risk")
Map.addLayer(diff_high, {}, "Sensitivity +20%")
Map.addLayer(diff_low, {}, "Sensitivity -20%")
Map

Map(center=[0, 0], controls=(WidgetControl(options=['position', 'transparent_bg'], widget=SearchDataGUI(childr…

In [ ]:
import ee
import geemap

ee.Authenticate()
ee.Initialize(project='ee-annsujitra001')

# -----------------------
# ROI: Nakhon Si Thammarat
# -----------------------
provinces = ee.FeatureCollection("FAO/GAUL/2015/level1")

roi = provinces.filter(ee.Filter.eq('ADM1_NAME', 'Nakhon Si Thammarat'))

# -----------------------
# Flood Risk Model
# -----------------------
# flood_risk already
# flood_risk

flood_class = flood_risk.expression(
    "(b(0) < 0.4) ? 1"
    ": (b(0) < 0.7) ? 2"
    ": 3"
).rename("risk").clip(roi)

In [ ]:
jrc = ee.Image("JRC/GSW1_4/GlobalSurfaceWater").select("occurrence")

# แปลงเป็น flood / non-flood (threshold)
# >50% = น้ำถาวร/น้ำท่วมบ่อย
water = jrc.gt(50).rename("water").clip(roi)


In [ ]:
pred = flood_class.eq(3).rename("pred")  # class 3 = high risk

In [ ]:
stack = pred.addBands(water)

samples = stack.stratifiedSample(
    numPoints=1000,
    classBand="water",
    region=roi,
    scale=100,
    geometries=True
)

In [ ]:
test = samples.randomColumn("rand")

train = test.filter(ee.Filter.lte("rand", 0.7))
val = test.filter(ee.Filter.gt("rand", 0.7))

conf_matrix = val.errorMatrix("water", "pred")

print("Confusion Matrix:", conf_matrix.getInfo())
print("Accuracy:", conf_matrix.accuracy().getInfo())
print("Kappa:", conf_matrix.kappa().getInfo())

Confusion Matrix: [[279, 24], [259, 34]]
Accuracy: 0.5251677852348994
Kappa: 0.037327062691201486


In [ ]:
Map = geemap.Map()
Map.centerObject(roi, 8)

Map.addLayer(flood_class,
             {"min":1, "max":3, "palette":["green","yellow","red"]},
             "Flood Risk")

Map.addLayer(water,
             {"palette":["blue"]},
             "JRC Water (Truth)")

Map

Map(center=[8.375329278730176, 99.78833952396523], controls=(WidgetControl(options=['position', 'transparent_b…

In [ ]:
import ee

# -----------------------
# Export JRC Water
# -----------------------

task = ee.batch.Export.image.toDrive(
    image=water,
    description='JRC_Water',
    folder='GEE',
    fileNamePrefix='JRC_Water',
    region=roi.geometry(),
    scale=30,
    maxPixels=1e13
)

task.start()
print("Export started!")

Export started!
